<a href="https://colab.research.google.com/github/zegavibiano22-tech/WiFi-CSI-CrossDomain-Sensing/blob/main/notebooks/%2002_baseline_cnn_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

# 1. Clone repositori jika foldernya belum ada di /content
if not os.path.exists('/content/WiFi-CSI-CrossDomain-Sensing'):
    !git clone https://github.com/zegavibiano22-tech/WiFi-CSI-CrossDomain-Sensing.git

# 2. Pindah ke direktori repositori
%cd /content/WiFi-CSI-CrossDomain-Sensing

# 3. Masukkan path ke sys.path agar modul 'src' bisa di-import langsung
if '/content/WiFi-CSI-CrossDomain-Sensing' not in sys.path:
    sys.path.append('/content/WiFi-CSI-CrossDomain-Sensing')

# 4. Hapus cache modul jika ada
if 'src.dataset' in sys.modules:
    del sys.modules['src.dataset']

# 5. Import modul dataset
from src.dataset import WidarBVPDataset
print("Berhasil mengimpor WidarBVPDataset terbaru!")

Cloning into 'WiFi-CSI-CrossDomain-Sensing'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 86 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 228.52 KiB | 10.88 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/WiFi-CSI-CrossDomain-Sensing
Berhasil mengimpor WidarBVPDataset terbaru!


Memastikan Colab masuk ke folder proyek WiFi-CSI-CrossDomain-Sensing, ngehapus cache kodingan dataset lama di memori, dan nge-import WidarBVPDataset versi paling baru dari src/dataset.py supaya bebas dari bug.

In [2]:
# 1. Clone repositori ke environment Colab
!git clone https://github.com/zegavibiano22-tech/WiFi-CSI-CrossDomain-Sensing.git
%cd WiFi-CSI-CrossDomain-Sensing

# 2. Import library utama dan modul custom
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import glob

from src.dataset import WidarBVPDataset

# Cek ketersediaan GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Cloning into 'WiFi-CSI-CrossDomain-Sensing'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 86 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 228.52 KiB | 10.88 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/WiFi-CSI-CrossDomain-Sensing/WiFi-CSI-CrossDomain-Sensing
Menggunakan device: cpu


Berfungsi mengunduh repositori dari GitHub ke lingkungan Colab, masuk ke dalam folder proyeknya, lalu memuat library PyTorch beserta modul WidarBVPDataset sekaligus mengecek ketersediaan GPU atau CPU untuk proses pelatihan.

In [3]:
class CNNLSTMBaseline(nn.Module):
    def __init__(self, num_classes=6):
        super(CNNLSTMBaseline, self).__init__()

        # 1. Feature Extractor (CNN) untuk tiap frame
        # Input shape per frame: (batch_size * 20, 1, 20, 20)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # Output: (32, 10, 10)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # Output: (64, 5, 5)
        )

        # Flatten CNN output: 64 * 5 * 5 = 1600
        self.fc_bridge = nn.Linear(64 * 5 * 5, 128)

        # 2. Temporal Feature Extractor (LSTM)
        # Input shape: (batch_size, time_steps=20, feature_dim=128)
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, num_layers=1, batch_first=True)

        # 3. Gesture Classifier (Output Layer)
        self.fc_classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 1, 20, 20, 20) -> (B, C, T, H, W)
        batch_size, channels, time_steps, height, width = x.size()

        # Merge batch & time dimensi untuk proses CNN paralel
        x = x.permute(0, 2, 1, 3, 4).contiguous() # (B, T, C, H, W)
        x = x.view(batch_size * time_steps, channels, height, width)

        # Ekstraksi fitur CNN
        cnn_out = self.cnn(x)
        cnn_out = cnn_out.view(cnn_out.size(0), -1) # Flatten
        features = self.fc_bridge(cnn_out)          # Shape: (B * T, 128)

        # Reshape kembali ke urutan waktu untuk LSTM
        features = features.view(batch_size, time_steps, -1) # Shape: (B, T, 128)

        # Proses LSTM
        lstm_out, (h_n, c_n) = self.lstm(features)

        # Ambil output dari frame/step terakhir
        last_time_step = lstm_out[:, -1, :] # Shape: (B, 64)

        # Klasifikasi gestur
        out_gesture = self.fc_classifier(last_time_step)

        return out_gesture

# Inisialisasi & uji Forward Pass dengan dummy data
model = CNNLSTMBaseline(num_classes=6).to(device)
dummy_input = torch.randn(2, 1, 20, 20, 20).to(device) # Batch size = 2
dummy_output = model(dummy_input)

print("Arsitektur Model CNN-LSTM Berhasil Dibuat!")
print("Shape Output Prediksi Gesture:", dummy_output.shape) # Expect: [2, 6]

Arsitektur Model CNN-LSTM Berhasil Dibuat!
Shape Output Prediksi Gesture: torch.Size([2, 6])


**Definisi arsitektur model CNN-LSTM untuk mengklasifikasikan gestur tangan dari data BVP.**

Prosesnya dibagi menjadi tiga tahapan utama:

2D-CNN (Ekstraksi Spasial): Memproses tiap frame data BVP satu per satu untuk mengekstrak bentuk dan pola gerakan spasialnya.

LSTM (Ekstraksi Temporal): Mempelajari urutan frame dari waktu ke waktu (20 frame) untuk memahami dinamika arah gerakan gesturnya.

Fully Connected Classifier (Klasifikasi): Mengambil hasil analisis urutan waktu dari LSTM untuk memprediksi salah satu dari 6 kelas gestur tangan.

Bagian paling bawah berfungsi untuk menguji model menggunakan data tiruan (dummy input) guna memastikan alur data dan ukuran output tensor-nya ([2, 6]) sudah sesuai sebelum dilatih dengan data asli.

In [4]:
import glob
import torch
from torch.utils.data import DataLoader

# 1. Cari semua file .mat secara rekursif di seluruh sub-folder /content/
all_files = glob.glob('/content/**/*.mat', recursive=True)

print(f"Total file .mat yang ditemukan: {len(all_files)}")

# 2. Logika penanganan jumlah file agar train_size tidak pernah 0
if len(all_files) == 0:
    raise RuntimeError("Tidak ada file .mat yang ditemukan! Pastikan file .mat sudah ada di /content/")

elif len(all_files) < 4:
    # Jika file kurang dari 4, gunakan semua file untuk train dan val agar tidak error
    train_files = all_files
    val_files = all_files
    batch_sz = 1
else:
    train_size = int(0.8 * len(all_files))
    val_size = len(all_files) - train_size
    train_files, val_files = torch.utils.data.random_split(all_files, [train_size, val_size])
    batch_sz = 4

# 3. Buat Dataset & DataLoader
train_dataset = WidarBVPDataset(train_files)
val_dataset = WidarBVPDataset(val_files)

train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)

print(f"Data Train: {len(train_dataset)} sampel | Data Val: {len(val_dataset)} sampel")

Total file .mat yang ditemukan: 1
Data Train: 1 sampel | Data Val: 1 sampel


Kode ini bertugas mengumpulkan semua file .mat di Colab, membaginya menjadi 80% data latih dan 20% data validasi, lalu membungkusnya ke dalam DataLoader dengan ukuran batch 4 agar siap dimasukkan ke model.

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    # Phase Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for bvp_batch, y_batch, _ in train_loader:
        bvp_batch, y_batch = bvp_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(bvp_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()

    train_acc = 100. * correct / total

    # Phase Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for bvp_batch, y_batch, _ in val_loader:
            bvp_batch, y_batch = bvp_batch.to(device), y_batch.to(device)
            outputs = model(bvp_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += y_batch.size(0)
            val_correct += predicted.eq(y_batch).sum().item()

    val_acc = 100. * val_correct / val_total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")

Epoch [1/10] Train Loss: 1.8388 | Train Acc: 0.00% | Val Loss: 1.6977 | Val Acc: 100.00%
Epoch [2/10] Train Loss: 0.9510 | Train Acc: 100.00% | Val Loss: 1.5665 | Val Acc: 100.00%
Epoch [3/10] Train Loss: 0.5612 | Train Acc: 100.00% | Val Loss: 1.4348 | Val Acc: 100.00%
Epoch [4/10] Train Loss: 0.3391 | Train Acc: 100.00% | Val Loss: 1.2900 | Val Acc: 100.00%
Epoch [5/10] Train Loss: 0.2284 | Train Acc: 100.00% | Val Loss: 1.1319 | Val Acc: 100.00%
Epoch [6/10] Train Loss: 0.1633 | Train Acc: 100.00% | Val Loss: 0.9639 | Val Acc: 100.00%
Epoch [7/10] Train Loss: 0.1214 | Train Acc: 100.00% | Val Loss: 0.7934 | Val Acc: 100.00%
Epoch [8/10] Train Loss: 0.0931 | Train Acc: 100.00% | Val Loss: 0.6338 | Val Acc: 100.00%
Epoch [9/10] Train Loss: 0.0732 | Train Acc: 100.00% | Val Loss: 0.4915 | Val Acc: 100.00%
Epoch [10/10] Train Loss: 0.0595 | Train Acc: 100.00% | Val Loss: 0.3700 | Val Acc: 100.00%


Kode ini adalah blok training loop yang berfungsi melatih dan menguji model selama 10 kali perulangan (epoch).

Proses utamanya dibagi menjadi dua tahap di setiap epoch:

Fase Pelatihan (Training): Model belajar memprediksi gestur dari data train, menghitung error (loss) menggunakan CrossEntropyLoss, lalu memperbarui bobot model menggunakan optimizer Adam agar prediksinya makin akurat.

Fase Evaluasi (Validation): Model diuji menggunakan data validation tanpa memperbarui bobot (torch.no_grad()) untuk melihat seberapa baik performa model dalam mengenali data baru.